In [1]:
# pip install cvzone

In [ ]:
import cv2
import cv2
import cvzone
from ultralytics import YOLO
import os
from collections import defaultdict

# =========================
# Load Models
# =========================

# Stage 1: Detect persons and motorcycles
detector = YOLO("../Models/yolo11m.pt")

# Stage 2: Helmet classifier
helmet_model = YOLO("../Models/helmet_violation.pt")

STAGE1_CLASSES = {0: "person", 3: "motorcycle"}
HELMET_CLASSES = {0: "With Helmet", 1: "Without Helmet"}

CONF_STAGE1 = 0.4
CONF_STAGE2 = 0.35

PADDING = 0.25

VIOLATION_COLOR = (0, 0, 255)
SAFE_COLOR = (0, 255, 0)

os.makedirs("Violations", exist_ok=True)
saved_ids = set()
helmet_memory = defaultdict(list)
saved_violations = []

# =========================
# Video Input
# =========================

video_path = "../Video_inpt/Helmet.mp4"

# Use FFmpeg backend
cap = cv2.VideoCapture(video_path, cv2.CAP_FFMPEG)

if not cap.isOpened():
    print("Error opening video")
    exit()

# =========================
# Video Output
# =========================

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fps = cap.get(cv2.CAP_PROP_FPS)

if fps == 0:
    fps = 20

fourcc = cv2.VideoWriter_fourcc(*'mp4v')

out = cv2.VideoWriter(
    "output3060(4).mp4",
    fourcc,
    fps,
    (frame_width, frame_height)
)

# =========================
# Helper Functions
# =========================

def expand_box(x1, y1, x2, y2, pad, img_h, img_w):
    w, h = x2 - x1, y2 - y1

    x1 = max(0, int(x1 - w * pad))
    y1 = max(0, int(y1 - h * pad))

    x2 = min(img_w, int(x2 + w * pad))
    y2 = min(img_h, int(y2 + h * pad))

    return x1, y1, x2, y2


def is_likely_rider(person_box, moto_boxes, iou_threshold=0.1):

    px1, py1, px2, py2 = person_box

    for mx1, my1, mx2, my2 in moto_boxes:

        inter_x1 = max(px1, mx1)
        inter_y1 = max(py1, my1)

        inter_x2 = min(px2, mx2)
        inter_y2 = min(py2, my2)

        if inter_x2 > inter_x1 and inter_y2 > inter_y1:

            inter_area = (inter_x2 - inter_x1) * (inter_y2 - inter_y1)

            person_area = (px2 - px1) * (py2 - py1)

            if inter_area / person_area > iou_threshold:
                return True

    return False


# =========================
# Main Loop
# =========================

while True:

    success, frame = cap.read()

    if not success or frame is None:
        break

    h, w = frame.shape[:2]

    # =========================
    # Stage 1 Detection
    # =========================

    stage1_results = detector.track(
        frame,
        persist=True,
        tracker="bytetrack.yaml",
        classes=[0, 3],
        conf=CONF_STAGE1
    )

    person_boxes = []
    moto_boxes = []

    for r in stage1_results:

        for box in r.boxes:

            cls = int(box.cls[0])

            x1, y1, x2, y2 = map(int, box.xyxy[0])

            track_id = -1
            if box.id is not None:
                track_id = int(box.id[0])

            if cls == 0:
                person_boxes.append((x1, y1, x2, y2, track_id))

            elif cls == 3:
                moto_boxes.append((x1, y1, x2, y2))

    # Draw motorcycle boxes
    for mx1, my1, mx2, my2 in moto_boxes:

        cv2.rectangle(
            frame,
            (mx1, my1),
            (mx2, my2),
            (200, 200, 0),
            1
        )

    # =========================
    # Stage 2 Helmet Detection
    # =========================

    for (px1, py1, px2, py2, track_id) in person_boxes:

        # Skip pedestrians
        if moto_boxes and not is_likely_rider(
            (px1, py1, px2, py2),
            moto_boxes
        ):
            continue

        # Expand ROI
        cx1, cy1, cx2, cy2 = expand_box(
            px1, py1, px2, py2,
            PADDING,
            h, w
        )

        crop = frame[cy1:cy2, cx1:cx2]

        if crop.size == 0:
            continue

        helmet_results = helmet_model(
            crop,
            conf=CONF_STAGE2
        )

        for hr in helmet_results:

            for hbox in hr.boxes:

                hcls = int(hbox.cls[0])
                hconf = float(hbox.conf[0])

                helmet_memory[track_id].append(hcls)
                if len(helmet_memory[track_id]) > 10:
                    helmet_memory[track_id].pop(0)

                final_cls = max(set(helmet_memory[track_id]), key=helmet_memory[track_id].count)

                label = HELMET_CLASSES.get(final_cls, "Unknown")
                color = SAFE_COLOR if final_cls == 0 else VIOLATION_COLOR

                # Helmet box
                hx1, hy1, hx2, hy2 = map(int, hbox.xyxy[0])

                # Convert to original frame coordinates
                hx1 += cx1
                hy1 += cy1
                hx2 += cx1
                hy2 += cy1

                display_pad = 20

                bx1 = max(0, hx1 - display_pad)
                by1 = max(0, hy1 - display_pad)
                bx2 = min(w, hx2 + display_pad)
                by2 = min(h, hy2 + display_pad)

                box_w = bx2 - bx1
                box_h = by2 - by1

                cvzone.cornerRect(
                    frame,
                    (bx1, by1, box_w, box_h),
                    l=25,
                    rt=4,
                    colorR=color
                )

                cvzone.putTextRect(
                    frame,
                    f"ID:{track_id} | {label} | {hconf:.2f}",
                    (max(0, bx1), max(50, by1)),
                    scale=2.0,
                    thickness=3,
                    colorR=color,
                    offset=12
                )

                if final_cls == 1:

                    nearest_moto = None
                    best_overlap = 0

                    for mx1, my1, mx2, my2 in moto_boxes:

                        ix1=max(px1,mx1)
                        iy1=max(py1,my1)
                        ix2=min(px2,mx2)
                        iy2=min(py2,my2)

                        if ix2>ix1 and iy2>iy1:
                            overlap=(ix2-ix1)*(iy2-iy1)
                            if overlap>best_overlap:
                                best_overlap=overlap
                                nearest_moto=(mx1,my1,mx2,my2)

                    if nearest_moto is not None:

                        mx1,my1,mx2,my2=nearest_moto

                        already_saved=False

                        for ox1,oy1,ox2,oy2 in saved_violations:

                            ix1=max(mx1,ox1)
                            iy1=max(my1,oy1)
                            ix2=min(mx2,ox2)
                            iy2=min(my2,oy2)

                            if ix2>ix1 and iy2>iy1:

                                inter=(ix2-ix1)*(iy2-iy1)
                                old_area=max(1,(ox2-ox1)*(oy2-oy1))

                                if inter/old_area > 0.5:
                                    already_saved=True
                                    break

                        if not already_saved:

                            saved_violations.append((mx1,my1,mx2,my2))

                            cv2.imwrite(
                                 f"Violations/violation_{track_id}.jpg",
                                    frame

                            )

                            # moto_crop=frame[my1:my2,mx1:mx2]

                            # if moto_crop.size > 0:
                            #     cv2.imwrite(
                            #         f"Violations/moto_{track_id}.jpg",
                            #         moto_crop
                            #     )

    # =========================
    # Save Frame
    # =========================

    out.write(frame)

    # =========================
    # Show Frame
    # =========================

    cv2.imshow("Helmet Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# =========================
# Release Resources
# =========================

cap.release()
out.release()

cv2.destroyAllWindows()

print("Video saved as output.mp4")

c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.2) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(



0: 384x640 14 persons, 10 motorcycles, 158.0ms
Speed: 2.5ms preprocess, 158.0ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 640x448 2 With Helmets, 187.8ms
Speed: 2.0ms preprocess, 187.8ms inference, 2.5ms postprocess per image at shape (1, 3, 640, 448)

0: 640x224 (no detections), 125.3ms
Speed: 1.6ms preprocess, 125.3ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 224)

0: 640x384 1 With Helmet, 152.1ms
Speed: 0.0ms preprocess, 152.1ms inference, 1.7ms postprocess per image at shape (1, 3, 640, 384)

0: 640x320 2 With Helmets, 112.2ms
Speed: 0.0ms preprocess, 112.2ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 320)

0: 640x416 1 With Helmet, 130.3ms
Speed: 1.0ms preprocess, 130.3ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 416)

0: 640x352 (no detections), 127.8ms
Speed: 0.0ms preprocess, 127.8ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 352)

0: 640x160 1 With Helmet, 73.3ms
Speed: 0.0ms pre

KeyboardInterrupt: 